# Exploring Odds API

### Imports

In [ ]:
import os
from dotenv import load_dotenv
import requests
import pandas as pd
import duckdb

In [ ]:
load_dotenv('../.env')
api_key = os.getenv('ODDS_API_KEY')
api_key is not None

In [ ]:
resp = requests.get(
    "https://api.the-odds-api.com/v4/sports/basketball_nba/odds",
    params={
        "apiKey": api_key,
        "regions": "uk,eu",
        "markets": "h2h",
        "oddsFormat": "decimal",
    },
)
resp.status_code, resp.headers.get("x-requests-remaining")

In [ ]:
data = resp.json()
len(data), data[0]

## Flattening Loop

In [ ]:
fetched_at = pd.Timestamp.now(tz='UTC')
rows = []

for event in data:
    for book in event['bookmakers']:
        for market in book['markets']:
            for outcome in market['outcomes']:
                rows.append({
                    'fetched_at' : fetched_at,
                    'event_id' : event['id'],
                    'commence_time' : event['commence_time'],
                    'home_team' : event['home_team'],
                    'away_team' : event['away_team'],
                    'bookmaker_key' : book['key'],
                    'market_key' : market['key'],
                    'outcome_name' : outcome['name'],
                    'price' : outcome['price'],
                })

In [ ]:
odds = pd.DataFrame(rows)
odds.shape

In [ ]:
odds.duplicated(subset=["fetched_at", "event_id", "bookmaker_key", "market_key", "outcome_name"]).any()

In [ ]:
odds[odds.bookmaker_key == "pinnacle"].head()

In [ ]:
odds.market_key.value_counts()

In [ ]:
odds.groupby("commence_time").bookmaker_key.nunique()

In [ ]:
odds[odds.commence_time == "2026-10-23T20:10:00Z"][["home_team", "away_team"]].drop_duplicates()

## Creating the Table

In [ ]:
con = duckdb.connect("../data/nba.duckdb")

In [ ]:
con.execute("""
            CREATE TABLE IF NOT EXISTS raw_odds(
                fetched_at  TIMESTAMPTZ,
                event_id    VARCHAR,
                commence_time   TIMESTAMPTZ,
                home_team   VARCHAR,
                away_team   VARCHAR,
                bookmaker_key   VARCHAR,
                market_key  VARCHAR,
                outcome_name    VARCHAR,
                price   DOUBLE,
                PRIMARY KEY     (fetched_at, event_id, bookmaker_key, market_key, outcome_name)
            )
            """)

In [ ]:
con.sql("DESCRIBE raw_odds")

In [ ]:
con.execute("INSERT INTO raw_odds BY NAME SELECT * FROM odds")
con.sql("SELECT COUNT(*) FROM raw_odds")